# 3.TCVectorDB带过滤的相似性搜索

知识点讲解：TCVectorDB 带元数据过滤的相似性搜索

1. TCVectorDB 与 Pinecone 元数据过滤的本质差异
   - Pinecone：无模式（schema-less）metadata，任意字段随写随用，
     过滤语法为类 MongoDB 的 JSON 字典（如 {"page": {"$gte": 5}}）
   - TCVectorDB：有模式（schema-based）metadata，必须预先声明字段类型，
     过滤语法为类 SQL 的字符串表达式（如 "page>=9"）
   - 这是使用 TCVectorDB 时最容易踩的坑：不声明 meta_fields 就无法过滤，
     而且往往不会报错，只是过滤条件被静默忽略或直接抛出字段不存在的异常

2. meta_fields 字段声明机制（本示例核心）
   - 作用：告诉 TCVectorDB 哪些标量字段需要建立索引以支持过滤
   - 用法：meta_fields=[MetaField(name="字段名", data_type=类型常量)]
   - 支持的类型常量（来自 langchain_community.vectorstores.tencentvectordb）：
     · META_FIELD_TYPE_STRING : 字符串，支持等值匹配与 in
     · META_FIELD_TYPE_UINT64 : 无符号 64 位整数，支持数值比较与范围查询
     · META_FIELD_TYPE_ARRAY  : 数组，支持包含关系判断
   - 课程文档强调：用非向量字段检索时，必须在构建 Collection 时添加对应索引，否则无法检索会报错
   - 重要限制：meta_fields 在 Collection 创建时生效并固化，
     集合建好后再改声明不会自动生效，需要新建集合

3. expr 过滤表达式语法（类 SQL）
   - 表达式格式：<field_name><operator><value>，多个表达式用 and / or / not 连接，可用括号分组
   - string 运算符：=、!=、in、not in（value 必须用英文双引号括起，如 doc_type="pdf"）
   - uint64 运算符：>、>=、=、<、<=、!=
   - 示例："page>=9"、"page in [1,3,5]"、"page=1 or page=10"、"(page>=5 and page<=8) or page=10"
   - 与 Pinecone 的 JSON 字典语法完全不同，迁移时这部分代码必须重写

4. 本示例的一个关键细节：embedding=None
   - 注意代码第 21 行创建了 OpenAIEmbeddings 实例，但第 23 行传的是 embedding=None
   - 这意味着本示例实际运行在「内置 Embedding 模式」下，
     上面创建的 embedding 变量并未被使用（属于示例代码的遗留）
   - 影响：向量化由腾讯云服务端完成，客户端不消耗 OpenAI 配额
   - 这也说明 meta_fields 过滤能力与 Embedding 模式是两个正交的维度，
     内置模式与外部模式都能使用元数据过滤

5. 元数据未声明字段的行为（本示例可观察到）
   - metadatas 中第 6 条额外带了 account_id 字段，但 meta_fields 只声明了 page
   - 结果：account_id 可以随数据写入存储，但无法出现在 expr 过滤条件中，
     若尝试 expr="account_id = 1" 会因字段无索引而失败
   - 对比 Pinecone：同样的数据在 Pinecone 中可以直接用 account_id 过滤

6. 典型输出示例与观察点
   - 执行 similarity_search_with_score("我养了一只猫，叫笨笨", expr="page>=9") 的返回：
       [('我的狗喜欢追逐球，看起来非常开心。', {'page': 10}, 0.757709),
        ('他们一起计划了一次周末的野餐，希望天气能好。', {'page': 9}, 0.675508)]
   - 观察点 1：query 问的是猫（「我养了一只猫，叫笨笨」），但语义最匹配的
     「笨笨是一只很喜欢睡觉的猫咪」（page=1）被过滤条件直接排除
   - 观察点 2：候选集只有 2 条，即使 k 默认为 4，最终也最多只能返回 2 条结果
   - 观察点 3：page=10 的「狗」排在 page=9 的「野餐」之前，因狗与猫同属宠物语义域
   - 结论：元数据过滤优先级高于语义相似度，条件过窄会严重损害召回

7. 最佳实践建议
   - 设计 Collection 时就把所有可能用于过滤的字段一次性声明完整，
     因为后期新增字段需要重建集合并重新灌数据，成本极高
   - 多租户隔离字段（account_id、tenant_id）必须声明为 UINT64 或 STRING 并建索引
   - 过滤表达式是字符串拼接，存在注入风险，
     用户输入必须做类型校验与转义，不要直接拼接到 expr 中
   - 与 Pinecone 一样，过滤条件过窄会导致候选集不足 k 条（本例 expr="page>=9" 仅 2 条）
   - 需要跨向量库通用的过滤逻辑时，考虑在应用层抽象一个过滤条件中间表示，
     再按目标数据库分别翻译为 JSON dict 或 SQL 表达式


In [ ]:
import os
import dotenv
from langchain_community.vectorstores import TencentVectorDB
from langchain_community.vectorstores.tencentvectordb import (
    ConnectionParams,
    MetaField,
    META_FIELD_TYPE_UINT64,
)
from langchain_openai import OpenAIEmbeddings
dotenv.load_dotenv()
embedding = OpenAIEmbeddings(model="text-embedding-3-small")
db = TencentVectorDB(
    embedding=None,
    connection_params=ConnectionParams(
        url=os.environ.get("TC_VECTOR_DB_URL"),
        username=os.environ.get("TC_VECTOR_DB_USERNAME"),
        key=os.environ.get("TC_VECTOR_DB_KEY"),
        timeout=int(os.environ.get("TC_VECTOR_DB_TIMEOUT")),
    ),
    database_name=os.environ.get("TC_VECTOR_DB_DATABASE"),
    collection_name="dataset-filter",
    meta_fields=[
        MetaField(name="page", data_type=META_FIELD_TYPE_UINT64),
    ]
)
texts = [
    "笨笨是一只很喜欢睡觉的猫咪",
    "我喜欢在夜晚听音乐，这让我感到放松。",
    "猫咪在窗台上打盹，看起来非常可爱。",
    "学习新技能是每个人都应该追求的目标。",
    "我最喜欢的食物是意大利面，尤其是番茄酱的那种。",
    "昨晚我做了一个奇怪的梦，梦见自己在太空飞行。",
    "我的手机突然关机了，让我有些焦虑。",
    "阅读是我每天都会做的事情，我觉得很充实。",
    "他们一起计划了一次周末的野餐，希望天气能好。",
    "我的狗喜欢追逐球，看起来非常开心。",
]
metadatas = [
    {"page": 1},
    {"page": 2},
    {"page": 3},
    {"page": 4},
    {"page": 5},
    {"page": 6, "account_id": 1},
    {"page": 7},
    {"page": 8},
    {"page": 9},
    {"page": 10},
]
ids = db.add_texts(texts, metadatas)
print("添加文档id列表:", ids)
print(db.similarity_search_with_score("我养了一只猫，叫笨笨", expr="page>=9"))